# Summary 19

# Домашнее задание 37
1. **Список всех стран**
> Используя базу данных world, выведите названия всех стран из таблицы country. Каждое название должно отображаться с новой строки и иметь номер.
Пример вывода: 
```
1. Aruba
2. Afghanistan
3. Angola
...  
239. Zimbabwe
```

In [ ]:
import pymysql

config = {'host': 'ich-db.edu.itcareerhub.de',
          'user': 'ich1',
          'password': 'password',
          'database': 'world',
          }

with pymysql.connect(**config) as connection:
    with connection.cursor() as cursor:
        cursor.execute("SELECT Name FROM country")
        for row in cursor:
            print(row[0])

In [ ]:
import pymysql

config = {'host': 'ich-db.edu.itcareerhub.de',
          'user': 'ich1',
          'password': 'password',
          'database': 'world',
          }

with pymysql.connect(**config) as connection:
    with connection.cursor() as cursor:
        cursor.execute("SELECT Name FROM country")
        countries = [row[0] for row in cursor]
        for i, name in enumerate(countries, start=1):
            print(f"{i}. {name}")


2. **Города выбранной страны**  
>Добавьте к предыдущей программе возможность выбора страны. Пусть пользователь должен ввести название страны. Далее выведите все города этой страны и их численность населения.
Пример вывода: 
```
Введите страну: Germany  
Berlin — 3386667
Hamburg — 1704735
Munich [München] — 1194560
...
```

In [ ]:
selected_country = input("Enter country: ")
with pymysql.connect(**config) as connection:
    with connection.cursor() as cursor:
        cursor.execute("""
            SELECT city.Name, city.Population
            FROM city
            JOIN country ON city.CountryCode = country.Code
            WHERE country.Name = %s
        """, (selected_country,))

        for res in cursor:
            print(f"{res[0]} — {res[1]}")

# Домашнее задание 38
1. **Добавление товаров**  
>Создайте программу, которая подключается к MongoDB и:
* выбирает базу ich_edit и коллекцию `products_<your_group>_<your_full_name>`
* очищает коллекцию перед началом
* добавляет 3 товара с полями: name, price, stock
* выводит сообщение о количестве добавленных товаров  
Пример вывода:   
3 products inserted.


In [ ]:
from pymongo import MongoClient

client = MongoClient(
    "mongodb://ich_editor:verystrongpassword"
    "@mongo.itcareerhub.de/?readPreference=primary"
    "&ssl=false&authMechanism=DEFAULT&authSource=ich_edit"
)

db = client["ich_edit"]
products = db["products_lesson_38"]

# Очистка перед добавлением
products.delete_many({})

items = [
    {"name": "Pen", "price": 1.5, "stock": 100},
    {"name": "Notebook", "price": 3.99, "stock": 50},
    {"name": "Backpack", "price": 25.0, "stock": 20},
]

result = products.insert_many(items)
print(f"{len(result.inserted_ids)} products inserted.")


2. **Увеличение цен**  
>Продолжите предыдущую задачу.  
>Теперь программа должна:
* увеличить цену всех товаров на 20%
* вывести количество обновлённых записей
* затем вывести список всех товаров с новыми ценами  
Пример вывода:   
Prices updated for 3 products.  
```
Updated products:
- Pen — $1.80
- Notebook — $4.79
- Backpack — $30.00
```

In [ ]:
from pymongo import MongoClient

client = MongoClient(
    "mongodb://ich_editor:verystrongpassword"
    "@mongo.itcareerhub.de/?readPreference=primary"
    "&ssl=false&authMechanism=DEFAULT&authSource=ich_edit"
)

db = client["ich_edit"]
products = db["products_lesson_38"]

# Увеличение цен на 20%
cursor = products.find({})
updated = 0

for doc in cursor:
    new_price = round(doc["price"] * 1.2, 2)
    result = products.update_one(
        {"_id": doc["_id"]},
        {"$set": {"price": new_price}}
    )
    if result.modified_count:
        updated += 1

print(f"Prices updated for {updated} products.\n")

# Вывод товаров после обновления
print("Updated products:")
for doc in products.find():
    print(f"- {doc['name']} — ${doc['price']:.2f}")
